In [1]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

True

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(
    model='gemini-2.0-flash-lite',
    temperature=0.7,
    max_tokens=None,
    timeout=None,
    max_retries=3,
    
)


c:\Users\NABEEL\.conda\envs\GenAI\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
os.getenv("SERPAPI_API_KEY")

## create the google seach tools

In [5]:
from langchain.utilities import SerpAPIWrapper
from langchain.tools import Tool
def get_serpapi_key():
    return os.getenv("SERPAPI_API_KEY")

def create_serpapi_search():
    search = SerpAPIWrapper(serpapi_api_key=get_serpapi_key())
    return search

def create_google_search_tool():
    search = create_serpapi_search()
    tool = Tool(
        name="Google Search",
        description="Useful for answering questions by searching Google.",
        func=search.run
    )
    return tool

google_search_tool = create_google_search_tool()

query = "Price of Gold Today in INR india?"
result = google_search_tool.run(query)

print("\nResult:\n", result)




Result:
 Gold rate in India today is ₹ 93,390 per 10 grams for 24 Carat and ₹ 85,550 for 22 Carat.


## Create weather info tool

In [6]:
#!pip install  requests
import requests
def get_weather_api():
    return os.getenv("OPENWEATHERMAP_API_KEY")

#weather search function
def weather_info(city_name):
    base_url = "https://api.openweathermap.org/data/2.5/weather"
    params = {
        "q": city_name,
        "appid": get_weather_api(),
        "units": "metric"  # for temperature in Celsius
    }
    response = requests.get(base_url, params=params)
    
    if response.status_code != 200:
        return f"Error fetching weather: {response.text}"
    
    data = response.json()
    temp = data["main"]["temp"]
    description = data["weather"][0]["description"]
    return f"The weather in {city_name} is {description} with a temperature of {temp}°C."

#create tool and pass function
def create_weather_tool():
    tool = Tool(
        name="Weather Checker",
        description="Useful for checking the current weather in a city. Input should be a city name.",
        func=weather_info
    )
    return tool

# now tool created ready to consume

#weather_tool = create_weather_tool()

query = "Pune"
result = create_weather_tool().run(query)

print("\nWeather Result:\n", result)



Weather Result:
 The weather in Pune is clear sky with a temperature of 39.55°C.


## Cricker live match API

In [7]:
# Import modules
import requests
import os
from langchain.tools import Tool

# Config - Cricket API Key
def get_cricket_api_key():
    return os.getenv("Cricket_API")


# Fetch live cricket score
def get_live_cricket_score():
    api_key = get_cricket_api_key()
    #print(api_key)
    url = f"https://api.cricapi.com/v1/currentMatches?apikey={api_key}&offset=0"

    response = requests.get(url)
    if response.status_code != 200:
        return f"Error fetching score: {response.text}"

    data = response.json()
    print(data)

    if not data.get("data"):
        return "No live matches found."

    results = []
    for match in data["data"]:
        if match["status"] == "live":
            team1 = match["teams"][0]
            team2 = match["teams"][1]
            score = match.get("score", [])
            status = match.get("status", "Unknown")

            results.append(f"{team1} vs {team2} - Status: {status}")

    if not results:
        return "No ongoing live matches at the moment."

    return "\n".join(results)

# Create Cricket Score Tool
def create_cricket_score_tool():
    tool = Tool(
        name="Live Cricket Score Checker",
        description="Useful for checking live cricket match scores. No input required.",
        func=lambda x: get_live_cricket_score()
    )
    return tool

# Use it!

cricket_tool = create_cricket_score_tool()

query = ""  # No input needed
result = cricket_tool.run(query)

print("\nCricket Score Result:\n", result)



{'apikey': '8713a673-15b4-4447-a649-e62938d4c0d9', 'data': [{'id': '15286b78-2a3d-4a4c-a3bb-62ed86d12712', 'name': 'Oman Women vs Bahrain Women, 1st T20I', 'matchType': 't20', 'status': 'Match not started', 'venue': 'Al Amerat Cricket Ground (Ministry Turf 2), Al Amerat', 'date': '2025-05-02', 'dateTimeGMT': '2025-05-02T04:00:00', 'teams': ['Oman Women', 'Bahrain Women'], 'score': [], 'series_id': 'e65792aa-6e4d-4c7d-8df5-534dd8160ec9', 'fantasyEnabled': False, 'bbbEnabled': False, 'hasSquad': True, 'matchStarted': True, 'matchEnded': False}, {'id': '52da6fcb-1e18-4a9f-a31e-7d762cce2670', 'name': 'Saudi Arabia vs Malaysia, Final', 'matchType': 't20', 'status': 'Malaysia won by 18 runs', 'venue': 'Bayuemas Oval, Kuala Lumpur', 'date': '2025-05-02', 'dateTimeGMT': '2025-05-02T06:00:00', 'teams': ['Saudi Arabia', 'Malaysia'], 'score': [{'r': 135, 'w': 7, 'o': 20, 'inning': 'Malaysia Inning 1'}, {'r': 117, 'w': 10, 'o': 19.2, 'inning': 'Saudi Arabia Inning 1'}], 'series_id': '043171a2-88f1

Here similarly I can create multiple tools as per need such as 
Football Score Tool
Stock Market Price Tool
Bitcoin Price Checker
Air Quality API Tool
News API Tool

Process:
create function that return api
create fultion that consume that API will do  tools task such as live score, stock market price etc
here one might need to do certain parsing it is very subjective to task what and how we have to do.
tip:- first print the whole output if after getting status code response 200
      how to use api check documentation and example availabe for majority of them

once it is done create the wrapper of tool and one can use @tool decorator
You can use above example as template
@tool is more useful with agent as per my understanding

Tool Type | Example

Built-in Tool | SerpAPI search, Wikipedia, PythonREPL

Custom Tool | Cricket Score, Weather API, Stock Prices API

## Create agent that will fetch the weather info and convert temp in c to f
## here I will use both Tool and @tool

In [15]:
import re
from langchain.tools import tool

@tool
def convert_c_to_f(temp_input: str) -> str:
    """
    Converts a temperature from Celsius to Fahrenheit.
    Input should be a string that includes the Celsius value (e.g., '15', '15°C', or full sentence).
    """
    # Extract the first number from the input string
    match = re.search(r'-?\d+(\.\d+)?', temp_input)
    if not match:
        return "Could not find a valid Celsius temperature in the input."
    
    temp_celsius = float(match.group())
    temp_fahrenheit = (temp_celsius * 9/5) + 32
    return f"{temp_celsius}°C is equal to {temp_fahrenheit:.2f}°F."


In [25]:
#since I already have created tool for weather I will use that only
from langchain.agents import initialize_agent, AgentType


tools = [create_weather_tool(),convert_c_to_f]

weather_info_agent = initialize_agent(tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True)
weather_agent = weather_info_agent.run('What is weather in Antarctica temp in Fahrenheit')
print(weather_agent)



> Entering new AgentExecutor chain...
I need to find the weather in Antarctica and then convert the temperature to Fahrenheit. First, I will check the weather in Antarctica.
Action: Weather Checker
Action Input: Antarctica
Observation: The weather in Antarctica is few clouds with a temperature of -44.18°C.
Thought:I now know the temperature in Antarctica is -44.18°C. I need to convert this to Fahrenheit.
Action: convert_c_to_f
Action Input: -44.18°C
Observation: -44.18°C is equal to -47.52°F.
Thought:I now know the final answer.
Final Answer: The temperature in Antarctica is -47.52°F.


> Finished chain.
The temperature in Antarctica is -47.52°F.


## Different Types of Agents
### AgentType.ZERO_SHOT_REACT_DESCRIPTION (Most used!)

    Default "ReAct" logic: Uses the prompt + descriptions of tools to decide what to do
    No training, works out of the box.
    ✅ Best when: You have clear tool descriptions and natural language queries.

### AgentType.REACT_DOCSTORE

    Uses ReAct logic but specifically for docstore-like tasks (e.g., document search).
    ⚠️ Obsolete in most modern use cases.
    ❌ Rarely used now.

### AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION

    Works like ZERO_SHOT but allows structured tool inputs (e.g., JSON-style params).
    ✅ Best when: You want to pass multiple structured parameters to a tool (e.g., {"city": "Delhi", "unit": "Celsius"}).

### AgentType.CONVERSATIONAL_REACT_DESCRIPTION

    Like ZERO_SHOT_REACT_DESCRIPTION but memory-aware.
    Tracks conversation history.
    ✅ Best for: Chatbots or multi-turn dialogue with context retention.

### AgentType.CHAT_CONVERSATIONAL_REACT_DESCRIPTION

    Combination of chat memory and tool selection.
    Similar to CONVERSATIONAL_REACT_DESCRIPTION but optimized for ChatOpenAI.


## Different methods available in Initialize_agent
run(input: str)
    
    Most used method.
    Executes the agent logic on a natural language input.
    Returns a string or final result.
    agent.run("What's the weather in Pune?")


.invoke(input: dict)

    More structured — passes a dictionary instead of a string.
    Useful if using memory or structured agents.
    agent.invoke({"input": "Tell me a joke."})


.stream(input: str)

    Yields intermediate steps (e.g., tool calls, reasoning).
    Best for real-time UIs or debugging.
    for step in agent.stream("Get the weather and convert it to Fahrenheit"):
        print(step)


.batch([input1, input2])

    Run multiple inputs in parallel.
    Best for performance and async processing
    agent.batch(["Weather in Delhi?", "Weather in London?"])
